Import des données - notebook d'exploration

In [5]:
import requests
import pandas as pd
import math
from pathlib import Path
import logging

On trouve les indicateur sur le site de la banque mondiale : 
- Population : data_worldbank_org/indicator/SP_POP_TOTL
- Superficie : https://data_worldbank_org/indicator/AG_SRF_TOTL_K2
- PIB (GDP)/hab : 
    GDP PPP :https://data_worldbank_org/indicator/NY_GDP_PCAP_PP_CD?most_recent_year_desc=true&year=2024

    GDP USD : https://data_worldbank_org/indicator/NY_GDP_PCAP_CD?most_recent_year_desc=true&year=2024
- Esperence de vie : https://data_worldbank_org/indicator/SP_DYN_LE00_IN
- CO2 par habitant : https://data_worldbank_org/indicator/EN_GHG_CO2_PC_CE_AR5
- PM2_5 : https://data_worldbank_org/indicator/EN_ATM_PM25_MC_M3
- urban population (%) : https://data_worldbank_org/indicator/SP_URB_TOTL_IN_ZS
- Pourcentage du PIB dans la santé : https://data_worldbank_org/indicator/SH_XPD_CHEX_GD_ZS

In [2]:
# Configuration de la requête 
year = 2024

#Dictionnaire des indicateurs

indicators = {
    "population": "SP_POP_TOTL",
    "Surface_area": "AG_LND_TOTL_K2",
    "gdp_per_capita_ppp_current": "NY_GDP_PCAP_PP_CD",
    "gdp_per_capita_usd_current": "NY_GDP_PCAP_CD",
    "life_expectancy": "SP_DYN_LE00_IN",
    "co2_emissions_per_capita": "EN_GHG_CO2_PC_CE_AR5",
    "pm2_5_exposure": "EN_ATM_PM25_MC_M3",
    'urban_population_percentage': "SP_URB_TOTL_IN_ZS",
    'health_expenditure_percentage_of_gdp': "SH_XPD_CHEX_GD_ZS",
}

database_id = "WB_WDI"

In [3]:
indicator_test = dict(list(indicators.items())[:1])
print(indicator_test)

{'population': 'SP_POP_TOTL'}


In [ ]:
def get_page(url, skip):

    try:
        response = requests.get(url,timeout=10, params={"skip" : skip})
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print("HTTP error occurred:", e)
    except requests.exceptions.RequestException as e:
        print("A request error occurred:", e)
        
    if response.status_code != 200:
            print(f"Erreur lors de la récupération des données : {response.status_code}")
            
    return response.json()

In [ ]:
def get_page(url, skip):
    response = requests.get(url, timeout=10, params={"skip": skip})
    response.raise_for_status()
    return response.json()

In [ ]:
def get_indicator_data(url, nom, pagination_limit):
    # first page data
    data = get_page(url, skip=0)
    n = math.ceil(data["count"]/pagination_limit)
    logging.info(f"Number of pages for {nom} : {n}")
    
    # other pages data
    for i in range(1,n):
        skip = i * pagination_limit
        data_page =get_page(url, skip=skip)
        data['value'].extend(data_page['value'])

    # catch the case where the number of items returned does not match the expected count
    if len(data['value']) != data['count']:
        raise ValueError(f"Expected {data['count']} items, but got {len(data['value'])}")

    # Transform into a DataFrame and select the desired columns
    df=pd.DataFrame(data['value'])
    nb_duplicates = df.duplicated().sum()
    if nb_duplicates != 0:
        raise ValueError(f"Doublons trouvés pour l'indicateur {nom} : {nb_duplicates}")

    return df

In [ ]:
start_year = 2000
end_year = 2024

pagination_limit = 1000  # Limite de pagination pour les requêtes
for nom, code in indicators.items():

    url = f"https://data360api.worldbank.org/data360/data?DATABASE_ID={database_id}&INDICATOR={database_id}_{code}&timePeriodFrom={start_year}&timePeriodTo={end_year}"

    try:
        df = get_indicator_data(url, nom, pagination_limit)
        df.to_csv(Path(f"../data/raw/{nom}_{start_year}-{end_year}_raw.csv"), index=False)
    
    except (requests.exceptions.RequestException, ValueError) as e:
        logging.error(f"Error for {nom}: {e}")
        continue 
     